In [0]:
from pyspark.sql.functions import *

bronze_df = spark.table("workspace.default.bronze_pet_store_records")

display(bronze_df)

In [0]:
silver_df = (
    bronze_df
        .dropDuplicates(["product_id"])
        .filter(col("price") > 0)
        .filter(col("sales") >= 0)
        .filter(col("rating").between(1,10))
        .withColumn("product_category", initcap(trim(col("product_category"))))
        .withColumn("country", upper(trim(col("country"))))
        .withColumn("pet_type", lower(trim(col("pet_type"))))
        .withColumn("pet_size", lower(trim(col("pet_size"))))
)

In [0]:
silver_df = (
    silver_df
        .withColumn("silver_timestamp", current_timestamp())
        .withColumn("pipeline_layer", lit("silver"))
)

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_pet_store_records")

In [0]:
silver = spark.table("workspace.default.silver_pet_store_records")

display(silver)

In [0]:
from pyspark.sql.functions import col, current_timestamp, lit

silver_df = spark.table("workspace.default.silver_pet_store_records")

silver_df = (
    silver_df
        .withColumn("estimated_revenue", col("sales") * col("price"))
        .withColumn("silver_timestamp", current_timestamp())
        .withColumn("pipeline_layer", lit("silver"))
)

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.silver_pet_store_records")

In [0]:
silver_df = spark.table("workspace.default.silver_pet_store_records")
silver_df.printSchema()